## Setup e caricamento artefatti

Importiamo le librerie e carichiamo:
- il modello addestrato (`.pkl`)
- la lista colonne del training (`train_columns.json`)

In [15]:
import pandas as pd
import numpy as np
import joblib
import json

## Caricamento modello e colonne di training

In [16]:
model = joblib.load("xgb_spaceship_model.pkl")

with open("train_columns.json") as f:
    train_columns = json.load(f)

print("Model loaded")
print("Number of train columns:", len(train_columns))

Model loaded
Number of train columns: 36


## Caricamento del dataset di test

Carichiamo `test.csv` (quello scaricato dalla competizione Kaggle).
Salviamo `PassengerId` perché servirà per creare il file di submission.

In [17]:
test_df = pd.read_csv("test.csv")

passenger_ids = test_df["PassengerId"].copy()

print("Test shape:", test_df.shape)
test_df.head()

Test shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


## Preprocessing (uguale al training)

Applichiamo gli stessi step usati nel notebook di training:
- split di Cabin in Deck/Num/Side
- drop Name (e PassengerId dalle feature, ma lo teniamo a parte)
- TotalSpending e NoSpending
- fill dei NaN

In [18]:
spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
categorical_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP", "CabinDeck", "CabinSide", "NoSpending"]

df_test_proc = test_df.copy()

# split Cabin
cabin_split = df_test_proc["Cabin"].str.split("/", expand=True)
df_test_proc["CabinDeck"] = cabin_split[0]
df_test_proc["CabinNum"]  = pd.to_numeric(cabin_split[1], errors="coerce")
df_test_proc["CabinSide"] = cabin_split[2]
df_test_proc = df_test_proc.drop(columns=["Cabin"])

# drop colonne inutili (PassengerId lo teniamo separato)
df_test_proc = df_test_proc.drop(columns=["Name"])

# TotalSpending
df_test_proc["TotalSpending"] = df_test_proc[spending_cols].sum(axis=1)

# NoSpending come stringa (coerente con one-hot)
df_test_proc["NoSpending"] = (df_test_proc["TotalSpending"] == 0).astype(str)

# fill spese
df_test_proc[spending_cols] = df_test_proc[spending_cols].fillna(0)

# Age e CabinNum
df_test_proc["Age"] = df_test_proc["Age"].fillna(df_test_proc["Age"].median())
df_test_proc["CabinNum"] = df_test_proc["CabinNum"].fillna(df_test_proc["CabinNum"].median())

# categoriche
for col in categorical_cols:
    df_test_proc[col] = df_test_proc[col].fillna("Unknown")

df_test_proc.isnull().sum().sort_values(ascending=False).head(10)

,0
PassengerId,0
HomePlanet,0
CryoSleep,0
Destination,0
Age,0
VIP,0
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0


## Creazione X_test + One-Hot + allineamento colonne

Ora creiamo le feature finali del test:
- rimuoviamo PassengerId (non è feature)
- applichiamo One-Hot Encoding
- allineiamo le colonne con quelle usate nel training (train_columns)

In [19]:
X_test = df_test_proc.drop(columns=["PassengerId"])

X_test = pd.get_dummies(X_test, drop_first=False)

# allinea colonne al train
X_test = X_test.reindex(columns=train_columns, fill_value=0)

print("X_test shape after alignment:", X_test.shape)
X_test.head()

X_test shape after alignment: (4277, 36)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,CabinNum,TotalSpending,HomePlanet_Earth,HomePlanet_Europa,...,CabinDeck_E,CabinDeck_F,CabinDeck_G,CabinDeck_T,CabinDeck_Unknown,CabinSide_P,CabinSide_S,CabinSide_Unknown,NoSpending_False,NoSpending_True
0,27.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,True,False,...,False,False,True,False,False,False,True,False,False,True
1,19.0,0.0,9.0,0.0,2823.0,0.0,4.0,2832.0,True,False,...,False,True,False,False,False,False,True,False,True,False
2,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,True,...,False,False,False,False,False,False,True,False,False,True
3,38.0,0.0,6652.0,0.0,181.0,585.0,1.0,7418.0,False,True,...,False,False,False,False,False,False,True,False,True,False
4,20.0,10.0,0.0,635.0,0.0,0.0,5.0,645.0,True,False,...,False,True,False,False,False,False,True,False,True,False


In [20]:
# forza l'allineamento ESATTO alle colonne del training
X_test = X_test.reindex(columns=train_columns, fill_value=0)

# controllo rapido: devono coincidere
print("Extra columns in X_test:", set(X_test.columns) - set(train_columns))
print("Missing columns in X_test:", set(train_columns) - set(X_test.columns))
print("Num columns X_test:", X_test.shape[1], "Num train_columns:", len(train_columns))

Extra columns in X_test: set()
Missing columns in X_test: set()
Num columns X_test: 36 Num train_columns: 36


## Predizione e creazione submission.csv

Prediciamo `Transported` sul test set e creiamo il file nel formato richiesto da Kaggle:
PassengerId, Transported

In [21]:
test_pred = model.predict(X_test)

# Kaggle vuole True/False
test_pred = test_pred.astype(bool)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Transported": test_pred
})

submission.to_csv("submission.csv", index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [22]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>